# Data enrichment: extracting and normalizing transfer recipients

**Goal:** turn the free-text `raw_description` of outgoing person-to-person transfers into a clean recipient name that can be compared across transactions.

**Why it is needed:** in the data exploration notebook I saw that the same recipient is written in several ways (for example `ZELLE PAYMENT TO KEVIN R`, `ZELLE TO KEVIN RICHARDS` and `WIRE TRANSFER TO KEVIN RICHARDS`). Without a clean name I can't tell whether money keeps going to the *same* person, which is the key to detecting patterns like escalating transfers to a new recipient or money-mule outflows.

**Approach, in two steps:**
1. **Extraction with an LLM (Gemini):** read each description and return the recipient name as structured JSON, together with a confidence score.
2. **Alias resolution with rules and fuzzy matching:** group the different spellings of the same person, per user, under a single `canonical_recipient`.

## 1. Setup: environment and credentials

**What I'm doing:** loading the environment variables from a `.env` file, reading which LLM provider to use (`LLM_PROVIDER`, `google` by default) and checking that the matching API key is present.

**Why:** the extraction calls an external API, so I don't want the key written in the notebook, and I prefer the notebook to stop right away with a clear message if the key is missing rather than failing halfway through a batch of calls. The saved output confirms that the provider is `google` and that the credentials were found.

In [46]:

import os
from dotenv import load_dotenv

# Carga las variables de entorno desde .env (buscando hacia arriba desde el cwd)
load_dotenv()

provider = os.environ.get("LLM_PROVIDER", "google")
print(f"Proveedor configurado: {provider}")

credenciales_esperadas = {
        "google": "GOOGLE_API_KEY"
    }
var_esperada = credenciales_esperadas.get(provider)
if var_esperada:
    assert os.environ.get(var_esperada), (
        f"Falta {var_esperada} en el .env para el proveedor '{provider}' — "
        f"copiá .env.example a .env y completá esa sección."
    )
    print("Credenciales OK.")


Proveedor configurado: google
Credenciales OK.


## 2. Helper to call the LLM

**What I'm doing:** defining `llamar_google`, a small wrapper around the Google Gemini client. It takes the prompt, an optional system instruction, the temperature and a maximum number of output tokens, and returns the response text.

**Why:** keeping the API call in one function means the rest of the notebook doesn't depend on the details of the client, and the model or its configuration can be changed in a single place.

In [47]:
from google import genai
from google.genai import types

def llamar_google(prompt, system=None, temperature=0.7, max_tokens=500):
    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        max_output_tokens=max_tokens,
    )
    resp = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=config,
    )
    return resp.text

## 3. System prompt for recipient extraction

**What I'm doing:** writing the system prompt that turns the model into an entity-extraction system. It states the task, a short list of rules and the exact output format, and includes four examples (few-shot).

**Why:** transfer descriptions are messy (`ZELLE PMT TO KEVIN R`, `WIRE TRANSFER D VOLKOV`), so I need an output that is **structured and predictable**. The main design choices are:
- **JSON only, no markdown**, so the answer can be parsed directly with `json.loads`.
- **Title case** for the name, so later comparisons are consistent.
- **Preserve abbreviations and never invent names** (`Kevin R` stays `Kevin R`). This keeps the LLM step conservative: completing partial names is left to the alias-resolution step later in the notebook, where the decision can be inspected.
- **Return `null` with a confidence score** when the recipient can't be identified, so descriptions that are not person-to-person transfers (like a store) don't produce made-up names.

**Heads-up:** the four examples in the prompt come from descriptions that also appear in this dataset (for example `ZELLE PAYMENT TO KEVIN R`), so those rows are not a fully independent test of the prompt.

In [48]:
SYSTEM_PROMPT = """
You are an entity extraction system for person-to-person bank transfers.

Your task is to extract and normalize the recipient name from a transfer description.

Rules:
1. Extract only the recipient name.
2. Normalize the name to title case.
3. Preserve abbreviations when the full name is not available.
4. If the recipient cannot be confidently identified, return null.
5. Do not invent or complete missing names.
6. Return valid JSON only.
7. Do not wrap the response in markdown.

Output format:

{
  "recipient_name": "Kevin Richards",
  "confidence": 0.98
}

Examples:

Input:
ZELLE PAYMENT TO KEVIN R

Output:
{
  "recipient_name": "Kevin R",
  "confidence": 0.95
}

Input:
ZELLE TO KEVIN RICHARDS

Output:
{
  "recipient_name": "Kevin Richards",
  "confidence": 0.99
}

Input:
WIRE TRANSFER D VOLKOV

Output:
{
  "recipient_name": "D Volkov",
  "confidence": 0.90
}

Input:
VENMO PAYMENT TO MARCO MARTINEZ

Output:
{
  "recipient_name": "Marco Martinez",
  "confidence": 0.99
}
"""


## 4. Keyword pre-filter for person-to-person transfers

**What I'm doing:** defining a list of keywords (`ZELLE`, `VENMO`, `WIRE TRANSFER`, `WIRE`) and a function, `needs_entity_extraction`, that returns `True` when a description contains any of them.

**Why:** only person-to-person transfers have a recipient worth extracting. Checking a keyword is free, while each LLM call costs time and money, so this filter would let me skip groceries, restaurants and other merchant transactions.

**Note:** this filter is only used in the draft at the end of the notebook (section 14). The main pipeline below selects its rows using `type` and `category` instead.

In [49]:
P2P_KEYWORDS = [
    "ZELLE",
    "VENMO",
    "WIRE TRANSFER",
    "WIRE",
]

def needs_entity_extraction(desc):
    desc = desc.upper()
    return any(k in desc for k in P2P_KEYWORDS)

## 5. Extraction function and a quick test

**What I'm doing:** wrapping the helper in `extraer_entidad`, which sends `Transaction description: <text>` together with the system prompt and uses `temperature=0`. Then I test it on three descriptions.

**Why:** `temperature=0` makes the model as deterministic as possible, which is what I want for extraction (there is nothing to be creative about). The three tests cover different cases: a **truncated name** (`KEVIN RICH`), a **merchant that is not a transfer** (`BOB EVANS #312 COLUMBUS OH`) and a **wire transfer** with the full name.

**What the saved output shows:** the truncated name is kept as `Kevin Rich` (confidence 0.98), the merchant returns `null` with confidence 0.0, and the wire transfer returns `Dmitri Volkov` (0.99). This matches the rules written in the prompt.

In [50]:
def extraer_entidad(descripcion):
    return llamar_google(
        prompt=f"Transaction description: {descripcion}",
        system=SYSTEM_PROMPT,
        temperature=0
    )

In [51]:
print(extraer_entidad("ZELLE PAYMENT TO KEVIN RICH"))
print(extraer_entidad("BOB EVANS #312 COLUMBUS OH"))
print(extraer_entidad("WIRE TRANSFER TO DMITRI VOLKOV"))

{
  "recipient_name": "Kevin Rich",
  "confidence": 0.98
}
{
  "recipient_name": null,
  "confidence": 0.0
}
{
  "recipient_name": "Dmitri Volkov",
  "confidence": 0.99
}


## 6. Loading the transactions

**What I'm doing:** reading `transactions.csv` from the project's `data` folder with pandas.

**Why:** it is the only table needed for this step, since the recipient information lives in `raw_description`.

In [ ]:
import pandas as pd
df_transactions = pd.read_csv('../data/transactions.csv')


## 7. Selecting the outgoing transfers

**What I'm doing:** keeping only the transactions that are **debits in the `transfer` category**, selecting a few columns (id, user, date, amount, description) and renaming some of them with an `outgoing_` prefix.

**Why:** these are the transactions where money leaves to another person, which is where recipient names matter. It leaves 37 rows, so calling the LLM once per row is affordable. The `outgoing_` prefix keeps the columns distinguishable if this table is later joined with other transaction tables (for example, incoming credits).

In [55]:
outgoing = (
    df_transactions[
        (df_transactions["type"] == "debit") &
        (df_transactions["category"] == "transfer")
    ]
    [["transaction_id", "user_id", "date", "amount", "raw_description"]]
    .rename(
        columns={
            "transaction_id": "outgoing_id",
            "date": "outgoing_date",
            "amount": "outgoing_amount"
        }
    )
)

## 8. Running the extraction on every outgoing transfer

**What I'm doing:** calling `extraer_entidad` once per row and storing the raw model answer in a new column, `entity_json`.

**Why:** I keep the raw JSON text first and parse it in a separate step. If the model ever returns something malformed, I can still inspect the original answer instead of losing it. The calls run one after another, which is fine for 37 rows.

In [56]:
outgoing["entity_json"] = outgoing["raw_description"].apply(extraer_entidad)

display(outgoing)

,outgoing_id,user_id,outgoing_date,outgoing_amount,raw_description,entity_json
25,TXN-522179,USR-001,2024-09-05 19:21:51,200.0,ZELLE PAYMENT TO KEVIN R,"{\n ""recipient_name"": ""Kevin R"",\n ""confiden..."
28,TXN-214576,USR-001,2024-09-12 11:12:12,350.0,ZELLE TO KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c..."
31,TXN-662336,USR-001,2024-09-18 15:08:27,500.0,ZELLE PAYMENT TO KEVIN RICH,"{\n ""recipient_name"": ""Kevin Rich"",\n ""confi..."
35,TXN-292401,USR-001,2024-09-25 12:29:15,750.0,ZELLE TRANSFER KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c..."
40,TXN-179046,USR-001,2024-10-02 15:51:55,1000.0,ZELLE PMT TO KEVIN R,"{\n ""recipient_name"": ""Kevin R"",\n ""confiden..."
46,TXN-997546,USR-001,2024-10-08 16:06:03,1500.0,ZELLE PAYMENT KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c..."
48,TXN-783823,USR-001,2024-10-15 16:53:00,2000.0,ZELLE TO KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c..."
52,TXN-197793,USR-001,2024-10-22 20:54:15,1200.0,WIRE TRANSFER TO KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c..."
57,TXN-789305,USR-002,2024-07-03 17:33:20,2000.0,ZELLE PAYMENT TO PATRICIA CHEN,"{\n ""recipient_name"": ""Patricia Chen"",\n ""co..."
69,TXN-956253,USR-002,2024-08-02 13:19:52,2000.0,ZELLE PAYMENT TO PATRICIA CHEN,"{\n ""recipient_name"": ""Patricia Chen"",\n ""co..."


## 9. Parsing the JSON into columns

**What I'm doing:** converting each `entity_json` string with `json.loads` and creating two columns: `recipient_name` and `confidence`.

**Why:** structured columns can be filtered, grouped and compared; a JSON string can't.

**What the output shows:** all 37 transfers got a name, with confidence between 0.95 and 0.99. Names from truncated descriptions stay truncated, exactly as the prompt asked, so the same person still appears under several spellings: `Kevin R`, `Kevin Rich` and `Kevin Richards`; `D Volkov`, `Dmitri V` and `Dmitri Volkov`; `Sofia M` and `Sofia Martinez`. The next sections deal with this.

In [57]:
import json

parsed = outgoing["entity_json"].apply(json.loads)

outgoing["recipient_name"] = parsed.apply(lambda x: x["recipient_name"])
outgoing["confidence"] = parsed.apply(lambda x: x["confidence"])

display(outgoing)

,outgoing_id,user_id,outgoing_date,outgoing_amount,raw_description,entity_json,recipient_name,confidence
25,TXN-522179,USR-001,2024-09-05 19:21:51,200.0,ZELLE PAYMENT TO KEVIN R,"{\n ""recipient_name"": ""Kevin R"",\n ""confiden...",Kevin R,0.95
28,TXN-214576,USR-001,2024-09-12 11:12:12,350.0,ZELLE TO KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c...",Kevin Richards,0.99
31,TXN-662336,USR-001,2024-09-18 15:08:27,500.0,ZELLE PAYMENT TO KEVIN RICH,"{\n ""recipient_name"": ""Kevin Rich"",\n ""confi...",Kevin Rich,0.98
35,TXN-292401,USR-001,2024-09-25 12:29:15,750.0,ZELLE TRANSFER KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c...",Kevin Richards,0.99
40,TXN-179046,USR-001,2024-10-02 15:51:55,1000.0,ZELLE PMT TO KEVIN R,"{\n ""recipient_name"": ""Kevin R"",\n ""confiden...",Kevin R,0.95
46,TXN-997546,USR-001,2024-10-08 16:06:03,1500.0,ZELLE PAYMENT KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c...",Kevin Richards,0.99
48,TXN-783823,USR-001,2024-10-15 16:53:00,2000.0,ZELLE TO KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c...",Kevin Richards,0.99
52,TXN-197793,USR-001,2024-10-22 20:54:15,1200.0,WIRE TRANSFER TO KEVIN RICHARDS,"{\n ""recipient_name"": ""Kevin Richards"",\n ""c...",Kevin Richards,0.99
57,TXN-789305,USR-002,2024-07-03 17:33:20,2000.0,ZELLE PAYMENT TO PATRICIA CHEN,"{\n ""recipient_name"": ""Patricia Chen"",\n ""co...",Patricia Chen,0.99
69,TXN-956253,USR-002,2024-08-02 13:19:52,2000.0,ZELLE PAYMENT TO PATRICIA CHEN,"{\n ""recipient_name"": ""Patricia Chen"",\n ""co...",Patricia Chen,0.99


## 10. Exploring string similarity between names

**What I'm doing:** for each distinct extracted name, using `rapidfuzz` (`process.extract`) to list the 5 most similar names and their score (0 to 100).

**Why:** before writing any matching rule I want to know whether plain string similarity can separate "same person, different spelling" from "different people", and which threshold would make sense.

**What the output shows:**
- Most pairs of spellings of the same person score between 82 and 90 (`Kevin R` ↔ `Kevin Richards` 90, `Dmitri V` ↔ `Dmitri Volkov` 90, `Sofia M` ↔ `Sofia Martinez` 90, `D Volkov` ↔ `Dmitri Volkov` 85.5).
- Different people who share a surname score around 70 or lower (`Marco Martinez` ↔ `Sofia Martinez` 71.4, `Isabella Martinez` ↔ `Marco Martinez` 69.1).
- Similarity alone is not enough, though: `Kevin Rich` ↔ `Kevin Richards` is only 83.3 and `D Volkov` ↔ `Dmitri V` just 37.5, both below a threshold of 85 even though they are the same person. I also need a rule based on first name and surname (next section).

In [ ]:
from rapidfuzz import process

names = outgoing["recipient_name"].dropna().unique()

for name in names:
    print(process.extract(name, names, limit=5))

[('Kevin R', 100.0, 0), ('Kevin Richards', 90.0, 1), ('Kevin Rich', 82.35294117647058, 2), ('Patricia Chen', 42.75, 3), ('Sofia Martinez', 36.642857142857146, 10)]
[('Kevin Richards', 100.0, 1), ('Kevin R', 90.0, 0), ('Kevin Rich', 83.33333333333334, 2), ('Patricia Chen', 42.22222222222222, 3), ('Sofia M', 36.642857142857146, 11)]
[('Kevin Rich', 100.0, 2), ('Kevin Richards', 83.33333333333334, 1), ('Kevin R', 82.35294117647058, 0), ('Patricia Chen', 41.30434782608695, 3), ('Sofia Martinez', 31.666666666666668, 10)]
[('Patricia Chen', 100.0, 3), ('Dmitri V', 45.0, 7), ('Kevin R', 42.75, 0), ('Kevin Richards', 42.22222222222222, 1), ('Kevin Rich', 41.30434782608695, 2)]
[('James Williams', 100.0, 4), ('Sofia M', 36.642857142857146, 11), ('Kevin Richards', 35.71428571428571, 1), ('Patricia Chen', 35.18518518518518, 3), ('Sofia Martinez', 33.92857142857142, 10)]
[('Dmitri Volkov', 100.0, 5), ('Dmitri V', 90.0, 7), ('D Volkov', 85.5, 6), ('Patricia Chen', 30.76923076923077, 3), ('Sofia Mar

## 11. Rules to decide whether two names are the same person

**What I'm doing:** defining two functions.
- `son_misma_persona(nombre1, nombre2)` returns `True` if two names probably belong to the same person. Identical names match immediately; missing values and single-word names never match. Otherwise each name is split into first name and last name, and the two names match if the first names are compatible (equal, one starts with the other, or same initial) **and** the last names are compatible in the same way, **or** if their overall similarity (`token_sort_ratio`) reaches the threshold (85 by default).
- `resolver_nombre(nombre, nombres_usuario)` collects every name in the list that matches the given one and returns the **longest** string, which is the most descriptive spelling (`Kevin Richards` rather than `Kevin R`).

**Why:** the previous section showed that similarity alone misses cases like `D Volkov` ↔ `Dmitri V`. Combining the first-name/last-name rule with the similarity score catches them, while requiring both first name and surname to be compatible keeps different people with the same surname (Marco, Sofia and Isabella Martinez) apart.

**Limitation:** the "same initial" shortcut is permissive. For example, `Marco Martinez` and `Maria Martinez` would be treated as the same person by this rule. It doesn't happen in this data, but it could on a larger dataset.

In [58]:
from rapidfuzz import fuzz


def son_misma_persona(nombre1, nombre2, threshold=85):
    """
    Determina si dos nombres probablemente representan
    la misma persona.
    """

    if pd.isna(nombre1) or pd.isna(nombre2):
        return False

    n1 = nombre1.strip().lower()
    n2 = nombre2.strip().lower()

    if n1 == n2:
        return True

    p1 = n1.split()
    p2 = n2.split()

    if len(p1) < 2 or len(p2) < 2:
        return False

    nombre_a, apellido_a = p1[0], p1[-1]
    nombre_b, apellido_b = p2[0], p2[-1]

    # mismo apellido o inicial del apellido
    apellido_match = (
        apellido_a == apellido_b
        or apellido_a.startswith(apellido_b)
        or apellido_b.startswith(apellido_a)
        or apellido_a[0] == apellido_b[0]
    )

    # mismo nombre o inicial del nombre
    nombre_match = (
        nombre_a == nombre_b
        or nombre_a.startswith(nombre_b)
        or nombre_b.startswith(nombre_a)
        or nombre_a[0] == nombre_b[0]
    )

    # similarity global
    similarity = fuzz.token_sort_ratio(n1, n2)

    return similarity >= threshold or (nombre_match and apellido_match)


def resolver_nombre(nombre, nombres_usuario, threshold=85):
    """
    Busca todos los aliases compatibles y devuelve
    el string más largo encontrado.
    """

    similares = []

    for otro in nombres_usuario:

        if pd.isna(otro):
            continue

        if son_misma_persona(nombre, otro, threshold):
            similares.append(otro)

    if len(similares) == 0:
        return nombre

    similares.append(nombre)

    # elegimos el más descriptivo
    return max(set(similares), key=len)

## 12. Resolving aliases per user

**What I'm doing:** for each user, collecting the distinct recipient names that user sent money to and replacing each name with its canonical version using `resolver_nombre`. The result goes in a new column, `canonical_recipient`, and I display the unique mapping.

**Why:** I do it **per user** so names are only compared against that user's own recipients: different users can legitimately send money to different people with similar names, and the spellings of a recipient are specific to each account.

**What the output shows:** 13 distinct extracted names collapse into 8 canonical recipients:
- `Kevin R`, `Kevin Rich` and `Kevin Richards` → **Kevin Richards**
- `D Volkov`, `Dmitri V` and `Dmitri Volkov` → **Dmitri Volkov**
- `Sofia M` → **Sofia Martinez**

The other five recipients (Patricia Chen, James Williams, Isabella Martinez, Marco Martinez and Susan Park) are unchanged, and the three different Martinez recipients of the same user stay separate.

In [59]:
outgoing["canonical_recipient"] = outgoing["recipient_name"]

THRESHOLD = 85

for user_id in outgoing["user_id"].unique():

    mask = outgoing["user_id"] == user_id

    nombres_usuario = (
        outgoing.loc[mask, "recipient_name"]
        .dropna()
        .unique()
        .tolist()
    )

    outgoing.loc[mask, "canonical_recipient"] = (
        outgoing.loc[mask, "recipient_name"]
        .apply(
            lambda nombre: resolver_nombre(
                nombre,
                nombres_usuario,
                threshold=THRESHOLD
            )
        )
    )

display(
    outgoing[
        [
            "user_id",
            "recipient_name",
            "canonical_recipient"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["user_id", "canonical_recipient"]
    )
)

,user_id,recipient_name,canonical_recipient
25,USR-001,Kevin R,Kevin Richards
28,USR-001,Kevin Richards,Kevin Richards
31,USR-001,Kevin Rich,Kevin Richards
57,USR-002,Patricia Chen,Patricia Chen
152,USR-003,Dmitri Volkov,Dmitri Volkov
155,USR-003,D Volkov,Dmitri Volkov
159,USR-003,Dmitri V,Dmitri Volkov
133,USR-003,James Williams,James Williams
184,USR-004,Isabella Martinez,Isabella Martinez
178,USR-004,Marco Martinez,Marco Martinez


## 13. Alternative compact version of the extraction (not executed)

**What I'm doing:** a shorter way to run the same extraction as sections 8 and 9: apply `extraer_entidad`, parse each answer with `json.loads` and expand the keys into columns with `pd.Series`, all in one chained expression.

**Heads-up:** this cell has no saved output. It is an **alternative** to sections 8 and 9, not a continuation: if it is run after them, it would call the API again for all 37 rows and would add a second copy of the `recipient_name` and `confidence` columns, which already exist.

In [ ]:
import json
import pandas as pd

resultados = (
    outgoing["raw_description"]
    .apply(extraer_entidad)
    .apply(json.loads)
    .apply(pd.Series)
)

outgoing = pd.concat([outgoing, resultados], axis=1)

display(outgoing)

## 15. Summary and next steps

**What the notebook produces:** a table with the 37 outgoing transfers, each with the extracted `recipient_name`, a `confidence` score and a `canonical_recipient` that groups the spellings of each person. In total, 37 transfers → 13 distinct extracted names → 8 canonical recipients, with confidence scores between 0.95 and 0.99.

**What it is for:** with `canonical_recipient` I can count payments per recipient, see when a recipient first appears and add up the amount sent to each one. That is what is needed to detect "new recipient with growing amounts" (Margaret's transfers to Kevin Richards) and repeated outflows to the same person after a deposit (Dorothy's transfers to Dmitri Volkov).

**Limitations**
- There are only 37 transfers, and the prompt examples overlap with the data, so the quality of the extraction on new data is untested.
- Even with `temperature=0`, an LLM can answer differently across runs or model versions, and the confidence score is reported by the model itself, not a calibrated probability.
- The alias rules are permissive about initials (section 11) and were tuned on this data.
- The enriched table only lives in memory: nothing saves it to disk.

**Possible next steps**
1. Cache the extraction by `raw_description`: many descriptions repeat (for example `ZELLE TO ISABELLA MARTINEZ`), so there is no need for one API call per row.
2. Fix the draft in section 14 so that only person-to-person transfers reach the API.
3. Save the enriched table and join `canonical_recipient` back to the transactions so it can be used in the exploration.
4. Check the extraction and the alias grouping against a hand-labeled sample.